# Updated colocation Mar 2026

- Replaces 0.5_socatv2024_colocation.ipynb
- Better to colocate SOCAT by comparison to the final coreINDEX since some profiles are dropped from open ocean masking


In [2]:
# os tools
import os
from tqdm import tqdm

# data tools
import numpy                 as np
import pandas                as pd
import xarray                as xr
from   datetime              import date, datetime, timedelta                 # for saving figures with today's date
import datetime
import scipy
from   scipy.stats           import kruskal              # for boxenplot stats
from   scipy.stats           import mannwhitneyu         # for split violin plot stats
import gsw


# for all plots
import matplotlib
import matplotlib.pyplot     as plt                      # needed to make map setup
import matplotlib.colors     as colors
from   matplotlib.ticker     import EngFormatter         # for degree symbol in axis
from   cmocean               import cm as cmo
import seaborn               as     sns

# for map
import shapefile
import cartopy                                           # to make map
import matplotlib.path       as     mpath                # to draw circle for map
import cartopy.crs           as     ccrs                 # for map projection
import cartopy.feature       as     cfeature             # to add land features to map

import pyproj  
import geopandas             as     gpd                  # for adding shapefiles of frontal zones 
from   osgeo                 import gdal


# Custom modules
import mod_cremas as crx 
import mod_ocean as myocn

from importlib import reload
import mod_plotting as myplt

import mod_loading as loader


# === CUSTOM FUNCTIONS
import mod_preprocessing as mod_prep
import mod_ocean

plt.rcParams.update(myplt.my_params(size=12))

In [3]:
import mod_argo

In [4]:
import shapefile
so_fronts = shapefile.Reader('./shapefiles/fronts/so_fronts.shp') 
stf_mod   = shapefile.Reader('./shapefiles/fronts/stf_mod/stf_mod.shp')

stf  = stf_mod.shape(0).points
saf  = so_fronts.shape(1).points
pf   = so_fronts.shape(2).points
sacc = so_fronts.shape(3).points
sie  = so_fronts.shape(4).points

max_latitude:          float = -30
add_gridlines:         bool  = True
color_land:            bool  = False
land_edgecolor:        str   = 'grey'
land_facecolor:        str   = 'grey'
fontsize:              float = 10
map_facecolor:         str   = 'white'
coast_linewidth:       float = 0.3
gridlines_linewidth:   float = 0.5
girdlines_color:       str   = 'grey'
gridlines_alpha:       float = 0.5
longitude_label_color: str   = 'grey'
latitude_label_color:  str   = 'grey'

# import data

- alternate socat processing


In [5]:
# this is the float data that will be clustered in 2.1
[coreDS, coreINDEX] = loader.import_core_data(type='processed') # has adt, sla, mld, pco2
[coreDS_test, coreINDEX_test] = loader.import_core_data(type='2024') # for testing with socat 2024 data, has adt, sla, mld, pco2
# [coreDS, _] = loader.import_core_data(type='L3_extended') # now includes core 2024 data for potential testing


In [6]:
coreDS = coreDS.reset_coords(['yearday', 'latitude', 'longitude', 'datetime'])
coreDS_test = coreDS_test.reset_coords(['yearday', 'latitude', 'longitude', 'datetime', 'year', 'month', 'day'])
combinedINDEX = xr.merge([coreINDEX, coreINDEX_test])

In [1]:
combinedINDEX

NameError: name 'combinedINDEX' is not defined

In [7]:
socatDS = loader.import_socat_L2(version='v2025') # includes socat 2024 test data

In [ ]:
# socat_trainval = pd.read_csv(folder + 'socat_trainval_processed_co2_mld_adt_yr2014-2024_acc20260326.csv', index_col=0)

In [31]:
argoDF.sort_values(by='weighted_dist').index.values[0]

'1901311_id092'

# Co-location with core argo data used for clustering

In [33]:
# troubleshooting new version
platDF = temp
argoINDEX = combinedINDEX
yd_window = 7
ref_time = '2014-01-01'
Lx=2.5


colocation = pd.DataFrame(index=platDF.index, columns=
                        ['nearest_profid', 'prof_datetime', 'prof_lat', 'prof_lon',
                            'yd_sep', 'km_sep'])
for idx in tqdm(platDF.index):
    lower = platDF.loc[idx, 'yearday'] - yd_window
    upper = platDF.loc[idx, 'yearday'] + yd_window
    argoset = (argoINDEX.where(argoINDEX.yearday > lower)
                .where(argoINDEX.yearday < upper)
                )

    argoDF = (argoset.reset_coords().to_dataframe() # .mean(dim='pressure')
                .dropna(subset = ['CT', 'SA'])
                )

    if len(argoDF) > 0:
        # Recalculate distance and yearday separation for each socat obs
        argoDF['yd_sep'] = argoDF.yearday.map(lambda x: abs(x - platDF.loc[idx, 'yearday']))
        argoDF['km_sep'] = argoDF.apply(lambda x: gsw.distance([platDF.loc[idx,'longitude'], x.longitude], 
                                                            [platDF.loc[idx,'latitude'], x.latitude])/1000, axis=1) # km

        argoDF['weighted_dist'] = argoDF.apply(lambda x: mod_argo.weighted_distance(
                                        [platDF.loc[idx,'longitude'], x.longitude], 
                                        [platDF.loc[idx,'latitude'], x.latitude], 
                                        Lx=Lx, Ly=1), axis=1)
        # imin = argoDF.idxmin(skipna=True).weighted_dist
        imin = argoDF.sort_values(by='weighted_dist').index.values[0]
        
        # Store metrics of nearest profile ID
        colocation.loc[idx, 'nearest_profid'] = imin
        colocation.loc[idx, 'prof_datetime'] = myocn.ytd2datetime(argoDF.loc[imin].yearday, ref_time=ref_time)
        colocation.loc[idx, 'prof_lat'] = argoDF.loc[imin].latitude
        colocation.loc[idx, 'prof_lon'] = argoDF.loc[imin].longitude
        colocation.loc[idx, 'yd_sep'] = argoDF.loc[imin].yd_sep
        colocation.loc[idx, 'km_sep'] = argoDF.loc[imin].km_sep


 35%|███▌      | 158/446 [00:25<00:47,  6.11it/s]


KeyboardInterrupt: 

In [34]:
savepath = '../working-vars/socat/colocate-coreArgo/' #(used to be in crusoe-repo)
save = True
datetag = datetime.datetime.now().strftime("%Y%m%d")

# ====
socat_df = socatDS.copy().to_dataframe()
colocate_7d_byYear = {k:None for k in [i for i in range(2014, 2025)]}

yrlist = [2014, 2015,
          2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
for yr in yrlist:
    print('===> Processing year:', yr)
    temp = socat_df[socat_df['datetime'].astype('datetime64[ns]').dt.year == yr].reset_index()  # be sure to reset index here, at end
    colocate_7d_byYear[yr] = mod_argo.find_nearest_float(temp, combinedINDEX, yd_window=7, ref_time='2014-01-01')

    if save:
        colocate_7d_byYear[yr].to_csv(savepath + 'colocate_v2025_validArgo_7d_yr' + str(yr) + '_acc' + datetag + '.csv')
        print('Saved to path:', savepath + 'colocate_v2025_validArgo_7d_yr' + str(yr) + '_acc' + datetag + '.csv')

===> Processing year: 2014


100%|██████████| 446/446 [01:08<00:00,  6.53it/s]


Saved to path: ../working-vars/socat/colocate-coreArgo/colocate_v2025_validArgo_7d_yr2014_acc20260327.csv
===> Processing year: 2015


100%|██████████| 699/699 [01:51<00:00,  6.27it/s]


Saved to path: ../working-vars/socat/colocate-coreArgo/colocate_v2025_validArgo_7d_yr2015_acc20260327.csv
===> Processing year: 2016


...

Saved to path: ../working-vars/socat/colocate-coreArgo/colocate_v2025_validArgo_7d_yr2021_acc20260327.csv
===> Processing year: 2022


100%|██████████| 396/396 [01:00<00:00,  6.53it/s]


Saved to path: ../working-vars/socat/colocate-coreArgo/colocate_v2025_validArgo_7d_yr2022_acc20260327.csv
===> Processing year: 2023


100%|██████████| 756/756 [01:51<00:00,  6.75it/s]

Saved to path: ../working-vars/socat/colocate-coreArgo/colocate_v2025_validArgo_7d_yr2023_acc20260327.csv


In [ ]:
for yr in [2024]:
    print('===> Processing year:', yr)
    temp = socat_df[socat_df['datetime'].astype('datetime64[ns]').dt.year == yr].reset_index()  # be sure to reset index here, at end
    colocate_7d_byYear[yr] = mod_argo.find_nearest_float(temp, combinedINDEX, yd_window=7, ref_time='2014-01-01')

    if save:
        colocate_7d_byYear[yr].to_csv(savepath + 'colocate_v2025_validArgo_7d_yr' + str(yr) + '_acc' + datetag + '.csv')
        print('Saved to path:', savepath + 'colocate_v2025_validArgo_7d_yr' + str(yr) + '_acc' + datetag + '.csv')

        # can be accessed using loader.import_socat_colocation() in future scripts

===> Processing year: 2024


100%|██████████| 434/434 [01:06<00:00,  6.55it/s]

Saved to path: ../working-vars/socat/colocate-coreArgo/colocate_v2025_validArgo_7d_yr2024_acc20260327.csv


In [ ]:
# Plot paneled, SOCAT obs by year
import mod_southpolarplot as sopo
reload(myocn)
temp = myocn.expand_datetime(socat_df, type='dataframe')

fig, axs = plt.subplots(2,5, figsize=(17, 10), layout='tight', subplot_kw={'projection': ccrs.SouthPolarStereo()})

for ax in axs.flatten():
  sopo.format_southpolar(ax,
                           add_gridlines = True,
                              color_land            = True,
                              land_facecolor        = land_facecolor,
                              land_edgecolor        = land_edgecolor,
                              fontsize              = 14,
                              max_latitude         = -35,
                              map_facecolor         = 'white',
                              coast_linewidth       = coast_linewidth)
  # sopo.add_frontlines(ax)

for ind, ax in enumerate(axs.flatten()):
    yr = ind + 2014
    data = temp[temp.year == yr]
    # data = temp.where(temp.year == yr)
    figtitle = str(yr)
    # print(len(data))
    ax.scatter(data.longitude, data.latitude, c='royalblue', s=5, alpha=0.85, transform=ccrs.PlateCarree(), label='socat', zorder=3)
    ax.set_title(figtitle)

# ax.legend()

# P1 processing: Process SOCAT fCO2 to pCO2

- Copied from 1.0_RUN_Preprocessing

In [ ]:
# Ner version Mar 2026 for SOCATv2025

sepstat_7d = loader.import_socat_colocation(buffer_time='7d')

# Moved to loader
# import os
# filepath = '../working-vars/socat/colocate-coreArgo/'
# sepdict_7d = {key:None for key in [str(x) for x in range(2014,2025)]}

# for x in os.listdir(filepath):
#     if x.startswith('colocate_v2025_validArgo_7d') & x.endswith('20260327.csv'):
#         # sepdict_7d[x[14:18]] = pd.read_csv(filepath+x, index_col=0)
#         sepdict_7d[x[30:34]] = pd.read_csv(filepath+x, index_col=0)
#         # print('Imported data for _7d window: ' + x)

# sepstat_7d = pd.concat(sepdict_7d.values()).reset_index().drop(['level_0', 'index'], axis=1)

In [42]:
sepstat_7d

,longitude,latitude,fco2rec,sal,sst,yearday,fco2water_equ_wet,fco2water_sst_wet,pco2water_equ_wet,pco2water_sst_wet,...,datetime,expoID,bathymetry,linear_time,nearest_profid,prof_datetime,prof_lat,prof_lon,yd_sep,km_sep
0,-24.64215,-73.72590,341.6865,34.0740,-1.2305,1.483947,NaN,341.700,NaN,343.2,...,2014-01-02 11:36:53,06AQ20131221_id0,-2062.0,1.483947,7900378_id041,2014-01-09,-71.616580,-22.823985,6.720278,242.124951
1,-36.40135,-77.10035,365.8905,34.2810,-1.7040,9.499508,NaN,365.900,NaN,367.5,...,2014-01-10 11:59:17,06AQ20131221_id0,-1087.0,9.499502,7900378_id041,2014-01-09,-71.616580,-22.823985,1.295284,729.272575
2,-38.14620,-77.89910,275.3920,34.2790,-1.4480,15.491701,NaN,275.400,NaN,276.6,...,2014-01-16 11:48:03,06AQ20131221_id0,-1168.0,15.491701,7900378_id042,2014-01-19,-71.682778,-23.213771,2.700012,811.679788
3,-38.94590,-77.60800,375.1900,34.4180,-1.7700,16.295203,NaN,375.200,NaN,376.9,...,2014-01-17 07:05:05,06AQ20131221_id0,-1009.0,16.295197,7900378_id042,2014-01-19,-71.682778,-23.213771,1.896510,799.773551
4,-28.47890,-74.55740,298.7880,33.7600,-1.3940,21.498843,NaN,298.800,NaN,300.1,...,2014-01-22 11:58:20,06AQ20131221_id1,-1639.0,21.498843,7900378_id043,2014-01-29,-71.743994,-23.090656,6.695984,357.493891
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7235,-136.38265,-73.29205,250.8455,33.2585,-1.1815,3702.136753,NaN,250.845,NaN,NaN,...,2024-02-20 03:16:55,WFCH20240206_id7,-3234.0,3702.136748,5906556_id041,2024-02-20,-70.554700,-135.543900,0.001024,305.744411
7236,-99.91180,-70.46775,398.7545,33.3240,-0.9140,3705.730804,NaN,398.755,NaN,NaN,...,2024-02-23 17:32:21,WFCH20240206_id11,-3693.0,3705.730799,3902325_id095,2024-02-24,-67.503000,-103.499490,0.303779,359.224182
7237,-65.76490,-60.64345,425.1250,33.7330,2.6230,3712.493576,NaN,425.125,NaN,NaN,...,2024-03-01 11:50:45,WFCH20240206_id13,-2631.0,3712.493576,4902638_id004,2024-03-05,-60.798679,-64.430321,4.092535,74.598890
7238,-55.87635,-44.48470,338.7715,34.0840,15.0715,3717.476510,NaN,338.770,NaN,NaN,...,2024-03-06 11:26:10,WFCH20240303_id0,-5326.0,3717.476505,3901565_id124,2024-03-12,-45.024100,-57.620200,6.053501,150.190991


In [43]:
# If starting new, import socat colocation data
# socat_colocated = loader.import_socat_colocation() # 7d 

socat_df = sepstat_7d[sepstat_7d['latitude']<-35]


In [ ]:
# ==== Convert SOCAT fugacity to partial pressure pCO2 (~1 min)
# Updated Feb 06 2026 # Jan 21 2026

shipDF = mod_prep.convert_socat_fco2(socat_df)
shipDF['mld'] = shipDF.nearest_profid.apply(lambda x: combinedINDEX.sel(profid=x).mld.values) #combined is core + core_test


In [45]:
combinedINDEX

<xarray.Dataset>
Dimensions:         (profid: 354858)
Coordinates:
  * profid          (profid) object '1900410_id260' ... '7902229_id004'
Data variables: (12/33)
    pressure        (profid) float64 6.4 7.0 6.0 6.6 5.9 ... 11.0 0.4 0.3 0.4
    wmoid           (profid) float64 1.9e+06 1.9e+06 ... 7.902e+06 7.902e+06
    latitude        (profid) float64 -40.36 -40.11 -40.25 ... -37.17 -36.95
    longitude       (profid) float64 95.36 96.08 96.68 ... 7.324 7.596 6.532
    datetime        (profid) object '2014-01-02 02:34:12.000000000' ... 17354...
    yearday         (profid) float64 1.107 11.27 21.44 ... 4.006e+03 4.015e+03
    ...              ...
    atmos_co2_ppm   (profid) float64 393.9 393.9 393.9 393.9 ... nan nan nan nan
    sst             (profid) float64 12.72 13.14 14.36 13.68 ... nan nan nan nan
    sss             (profid) float64 35.2 35.2 35.14 35.18 ... nan nan nan nan
    vapor_pres_atm  (profid) float64 0.01422 0.01462 0.01582 ... nan nan nan
    pco2_atmos      (profid) float64 392.0 391.1 386.1 389.0 ... nan nan nan nan
    mlp             (profid) float64 nan nan nan nan ... 12.37 30.27 50.96 29.54

In [46]:
shipDF = shipDF.loc[:,~shipDF.columns.duplicated()].copy() # if linear_time got duplicated
shipDF = mod_prep.add_regression_time_vars(shipDF)
shipDF = mod_prep.add_regression_carbon_vars(shipDF, convert_pco2=True) 
shipDF['delta_pco2'] = shipDF['pco2_ocean'] - shipDF['pco2_atmos']
# 
# shipDF

/Users/sangminsong/Library/CloudStorage/OneDrive-UW/Code/CRUSOE/src/mod_ocean.py:35: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  (df['datetime'].dt.is_leap_year.replace({True: 366, False: 365}))


In [50]:
# Add ADT and add profid to soccom_df
print('Adding satellite ADT...')
shipDF_adt = mod_prep.add_satellite_adt(shipDF, year_range = range(2014,2025))

Adding satellite ADT...


In [66]:
shipDF_adt

,longitude,latitude,fco2rec,sal,sst,linear_time,fco2water_equ_wet,fco2water_sst_wet,pco2water_equ_wet,pco2water_sst_wet,...,atmos_co2_ppm,sss,vapor_pres_atm,pco2_atmos,delta_pco2,mld,year,month,day,adt
0,-24.64215,-73.72590,341.6865,34.0740,-1.2305,1.483947,NaN,341.700,NaN,343.2,...,393.99634972925736,34.405349,0.005405,385.975982,-42.756075,11.667522,2014,1,2,-1.2091
1,-36.40135,-77.10035,365.8905,34.2810,-1.7040,9.499502,NaN,365.900,NaN,367.5,...,393.92330529750296,34.614034,0.005219,385.238844,-17.695680,11.667522,2014,1,10,-0.9939
2,-38.14620,-77.89910,275.3920,34.2790,-1.4480,15.491701,NaN,275.400,NaN,276.6,...,393.84300194956154,34.612038,0.005318,383.955134,-107.323573,10.548086,2014,1,16,-0.9822
3,-38.94590,-77.60800,375.1900,34.4180,-1.7700,16.295197,NaN,375.200,NaN,376.9,...,393.8420320263694,34.752420,0.005193,384.469845,-7.583650,10.548086,2014,1,17,-1.0041
4,-28.47890,-74.55740,298.7880,33.7600,-1.3940,21.498843,NaN,298.800,NaN,300.1,...,393.75378587854846,34.088126,0.005341,390.543232,-90.411353,10.657734,2014,1,22,-1.2278
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7235,-136.38265,-73.29205,250.8455,33.2585,-1.1815,3702.136748,NaN,250.845,NaN,NaN,...,417.54551391571215,33.580067,0.005426,411.097051,-159.126567,30.405754,2024,2,20,-1.1179
7236,-99.91180,-70.46775,398.7545,33.3240,-0.9140,3705.730799,NaN,398.755,NaN,NaN,...,417.5354155547392,33.643676,0.005534,400.987695,-0.451374,49.797021,2024,2,23,-1.0880
7237,-65.76490,-60.64345,425.1250,33.7330,2.6230,3712.493576,NaN,425.125,NaN,NaN,...,417.48524665126774,34.055110,0.007141,399.609350,27.326695,42.749931,2024,3,1,-0.8436
7238,-55.87635,-44.48470,338.7715,34.0840,15.0715,3717.476505,NaN,338.770,NaN,NaN,...,417.61437556576453,34.406736,0.016576,413.803880,-73.805735,21.728755,2024,3,6,-0.0622


# temporary save point

In [8]:
temp_2024 = pd.read_csv('../working-vars/regression/P1-processed/socatv2025_test_processed_co2_mld_adt_yr2024_acc20260327.csv', index_col=0)
temp_all = pd.read_csv('../working-vars/regression/P1-processed/socatv2025_trainval_processed_co2_mld_adt_yr2014-2023_acc20260327.csv', index_col=0)

In [12]:
temp_all

,cruiseid,datetime,longitude,latitude,linear_time,nearest_profid,prof_datetime,prof_lat,prof_lon,yd_sep,...,sss,mld,adt,atmos_pres_Pa,atmos_pres_atm,vapor_pres_atm,fco2rec,pco2_atmos,pco2_ocean,delta_pco2
0,06AQ20131221,2014-01-02 11:36:53,-24.64215,-73.72590,1.483947,7900378_id041,2014-01-09,-71.616580,-22.823985,6.720278,...,34.405349,11.667522,-1.2091,99810.0,0.985048,0.005405,341.6865,385.975982,343.219907,-42.756075
1,06AQ20131221,2014-01-10 11:59:17,-36.40135,-77.10035,9.499502,7900378_id041,2014-01-09,-71.616580,-22.823985,1.295284,...,34.614034,11.667522,-0.9939,99620.0,0.983173,0.005219,365.8905,385.238844,367.543164,-17.695680
2,06AQ20131221,2014-01-16 11:48:03,-38.14620,-77.89910,15.491701,7900378_id042,2014-01-19,-71.682778,-23.213771,2.700012,...,34.612038,10.548086,-0.9822,99320.0,0.980212,0.005318,275.3920,383.955134,276.631562,-107.323573
3,06AQ20131221,2014-01-17 07:05:05,-38.94590,-77.60800,16.295197,7900378_id042,2014-01-19,-71.682778,-23.213771,1.896510,...,34.752420,10.548086,-1.0041,99440.0,0.981396,0.005193,375.1900,384.469845,376.886195,-7.583650
4,06AQ20131221,2014-01-22 11:58:20,-28.47890,-74.55740,21.498843,7900378_id043,2014-01-29,-71.743994,-23.090656,6.695984,...,34.088126,10.657734,-1.2278,101040.0,0.997187,0.005341,298.7880,390.543232,300.131879,-90.411353
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6801,WFCH20231213,2023-12-15 12:29:15,-68.07260,-62.53640,3635.520312,5906560_id034,2023-12-19,-62.277300,-68.640400,4.288443,...,34.176978,50.752939,-0.8102,98320.0,0.970343,0.006176,405.2335,402.698972,407.007483,4.308512
6802,WFCH20231213,2023-12-25 13:32:16,-65.75710,-62.50970,3645.564074,5906560_id035,2023-12-30,-62.152700,-68.108100,4.586534,...,34.209054,45.531411,-0.9850,97010.0,0.957414,0.006317,404.7990,397.214424,406.563603,9.349179
6803,WFCH20231213,2023-12-26 08:47:44,-66.04485,-57.50905,3646.366481,5906560_id035,2023-12-30,-62.152700,-68.108100,3.784126,...,34.370827,45.531411,0.2052,98600.0,0.973106,0.008910,383.6670,402.619480,385.234157,-17.385323
6804,WFCH20231227,2023-12-29 17:03:41,-64.97110,-56.32710,3649.710891,6903769_id146,2024-01-04,-58.725878,-69.085745,5.560289,...,34.388894,72.831037,-0.0195,98620.0,0.973304,0.008811,386.5280,402.644772,388.110232,-14.534540


In [9]:
shipDF_adt = pd.concat([temp_all, temp_2024])
shipDF_adt

,cruiseid,datetime,longitude,latitude,linear_time,nearest_profid,prof_datetime,prof_lat,prof_lon,yd_sep,...,sss,mld,adt,atmos_pres_Pa,atmos_pres_atm,vapor_pres_atm,fco2rec,pco2_atmos,pco2_ocean,delta_pco2
0,06AQ20131221,2014-01-02 11:36:53,-24.64215,-73.72590,1.483947,7900378_id041,2014-01-09,-71.616580,-22.823985,6.720278,...,34.405349,11.667522,-1.2091,99810.0,0.985048,0.005405,341.6865,385.975982,343.219907,-42.756075
1,06AQ20131221,2014-01-10 11:59:17,-36.40135,-77.10035,9.499502,7900378_id041,2014-01-09,-71.616580,-22.823985,1.295284,...,34.614034,11.667522,-0.9939,99620.0,0.983173,0.005219,365.8905,385.238844,367.543164,-17.695680
2,06AQ20131221,2014-01-16 11:48:03,-38.14620,-77.89910,15.491701,7900378_id042,2014-01-19,-71.682778,-23.213771,2.700012,...,34.612038,10.548086,-0.9822,99320.0,0.980212,0.005318,275.3920,383.955134,276.631562,-107.323573
3,06AQ20131221,2014-01-17 07:05:05,-38.94590,-77.60800,16.295197,7900378_id042,2014-01-19,-71.682778,-23.213771,1.896510,...,34.752420,10.548086,-1.0041,99440.0,0.981396,0.005193,375.1900,384.469845,376.886195,-7.583650
4,06AQ20131221,2014-01-22 11:58:20,-28.47890,-74.55740,21.498843,7900378_id043,2014-01-29,-71.743994,-23.090656,6.695984,...,34.088126,10.657734,-1.2278,101040.0,0.997187,0.005341,298.7880,390.543232,300.131879,-90.411353
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7235,WFCH20240206,2024-02-20 03:16:55,-136.38265,-73.29205,3702.136748,5906556_id041,2024-02-20,-70.554700,-135.543900,0.001024,...,33.580067,30.405754,-1.1179,100310.0,0.989983,0.005426,250.8455,411.097051,251.970483,-159.126567
7236,WFCH20240206,2024-02-23 17:32:21,-99.91180,-70.46775,3705.730799,3902325_id095,2024-02-24,-67.503000,-103.499490,0.303779,...,33.643676,49.797021,-1.0880,97870.0,0.965902,0.005534,398.7545,400.987695,400.536321,-0.451374
7237,WFCH20240206,2024-03-01 11:50:45,-65.76490,-60.64345,3712.493576,4902638_id004,2024-03-05,-60.798679,-64.430321,4.092535,...,34.055110,42.749931,-0.8436,97710.0,0.964323,0.007141,425.1250,399.609350,426.936046,27.326695
7238,WFCH20240303,2024-03-06 11:26:10,-55.87635,-44.48470,3717.476505,3901565_id124,2024-03-12,-45.024100,-57.620200,6.053501,...,34.406736,21.728755,-0.0622,102080.0,1.007451,0.016576,338.7715,413.803880,339.998145,-73.805735


In [10]:
shipDF_wind = mod_prep.add_wind_speed(shipDF_adt, year_range = range(2014,2025))


# temp['sea_ice_conc'] = temp.apply(lambda row: get_nearest_seaice(row), axis=1)

100%|██████████| 11/11 [02:55<00:00, 15.92s/it]


In [11]:
shipDF_wind.to_csv('../working-vars/regression/P1-processed/socatv2025_temporary_windspeed.csv')
shipDF_wind

,cruiseid,datetime,longitude,latitude,linear_time,nearest_profid,prof_datetime,prof_lat,prof_lon,yd_sep,...,vapor_pres_atm,fco2rec,pco2_atmos,pco2_ocean,delta_pco2,year,month,day,wind_components,wind_speed
0,06AQ20131221,2014-01-02 11:36:53,-24.64215,-73.72590,1.483947,7900378_id041,2014-01-09,-71.616580,-22.823985,6.720278,...,0.005405,341.6865,385.975982,343.219907,-42.756075,2014,1,2,"(3.2225647, 3.5028076)",4.759683
1,06AQ20131221,2014-01-10 11:59:17,-36.40135,-77.10035,9.499502,7900378_id041,2014-01-09,-71.616580,-22.823985,1.295284,...,0.005219,365.8905,385.238844,367.543164,-17.695680,2014,1,10,"(-2.2945557, -1.2818146)",2.628314
2,06AQ20131221,2014-01-16 11:48:03,-38.14620,-77.89910,15.491701,7900378_id042,2014-01-19,-71.682778,-23.213771,2.700012,...,0.005318,275.3920,383.955134,276.631562,-107.323573,2014,1,16,"(-0.820755, -2.346283)",2.485696
3,06AQ20131221,2014-01-17 07:05:05,-38.94590,-77.60800,16.295197,7900378_id042,2014-01-19,-71.682778,-23.213771,1.896510,...,0.005193,375.1900,384.469845,376.886195,-7.583650,2014,1,17,"(-0.9001007, 5.141922)",5.220109
4,06AQ20131221,2014-01-22 11:58:20,-28.47890,-74.55740,21.498843,7900378_id043,2014-01-29,-71.743994,-23.090656,6.695984,...,0.005341,298.7880,390.543232,300.131879,-90.411353,2014,1,22,"(3.8532562, -2.301178)",4.488096
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7235,WFCH20240206,2024-02-20 03:16:55,-136.38265,-73.29205,3702.136748,5906556_id041,2024-02-20,-70.554700,-135.543900,0.001024,...,0.005426,250.8455,411.097051,251.970483,-159.126567,2024,2,20,"(-6.5198975, -0.71287537)",6.558754
7236,WFCH20240206,2024-02-23 17:32:21,-99.91180,-70.46775,3705.730799,3902325_id095,2024-02-24,-67.503000,-103.499490,0.303779,...,0.005534,398.7545,400.987695,400.536321,-0.451374,2024,2,23,"(12.030548, -2.8300934)",12.358945
7237,WFCH20240206,2024-03-01 11:50:45,-65.76490,-60.64345,3712.493576,4902638_id004,2024-03-05,-60.798679,-64.430321,4.092535,...,0.007141,425.1250,399.609350,426.936046,27.326695,2024,3,1,"(7.3894653, -3.107254)",8.016185
7238,WFCH20240303,2024-03-06 11:26:10,-55.87635,-44.48470,3717.476505,3901565_id124,2024-03-12,-45.024100,-57.620200,6.053501,...,0.016576,338.7715,413.803880,339.998145,-73.805735,2024,3,6,"(-0.095825195, -2.4891968)",2.491040


In [20]:
shipDF_ice = mod_prep.add_sea_ice(shipDF_wind, year_range = range(2014,2025))


100%|██████████| 11/11 [00:07<00:00,  1.57it/s]


In [21]:
shipDF_ice.to_csv('../working-vars/regression/P1-processed/socatv2025_temporary_icefraction.csv')
shipDF_ice

,cruiseid,datetime,longitude,latitude,linear_time,nearest_profid,prof_datetime,prof_lat,prof_lon,yd_sep,...,fco2rec,pco2_atmos,pco2_ocean,delta_pco2,year,month,day,wind_components,wind_speed,sea_ice
0,06AQ20131221,2014-01-02 11:36:53,-24.64215,-73.72590,1.483947,7900378_id041,2014-01-09,-71.616580,-22.823985,6.720278,...,341.6865,385.975982,343.219907,-42.756075,2014,1,2,"(3.2225647, 3.5028076)",4.759683,0.27
1,06AQ20131221,2014-01-10 11:59:17,-36.40135,-77.10035,9.499502,7900378_id041,2014-01-09,-71.616580,-22.823985,1.295284,...,365.8905,385.238844,367.543164,-17.695680,2014,1,10,"(-2.2945557, -1.2818146)",2.628314,0.82
2,06AQ20131221,2014-01-16 11:48:03,-38.14620,-77.89910,15.491701,7900378_id042,2014-01-19,-71.682778,-23.213771,2.700012,...,275.3920,383.955134,276.631562,-107.323573,2014,1,16,"(-0.820755, -2.346283)",2.485696,0.73
3,06AQ20131221,2014-01-17 07:05:05,-38.94590,-77.60800,16.295197,7900378_id042,2014-01-19,-71.682778,-23.213771,1.896510,...,375.1900,384.469845,376.886195,-7.583650,2014,1,17,"(-0.9001007, 5.141922)",5.220109,0.83
4,06AQ20131221,2014-01-22 11:58:20,-28.47890,-74.55740,21.498843,7900378_id043,2014-01-29,-71.743994,-23.090656,6.695984,...,298.7880,390.543232,300.131879,-90.411353,2014,1,22,"(3.8532562, -2.301178)",4.488096,0.39
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7235,WFCH20240206,2024-02-20 03:16:55,-136.38265,-73.29205,3702.136748,5906556_id041,2024-02-20,-70.554700,-135.543900,0.001024,...,250.8455,411.097051,251.970483,-159.126567,2024,2,20,"(-6.5198975, -0.71287537)",6.558754,0.0
7236,WFCH20240206,2024-02-23 17:32:21,-99.91180,-70.46775,3705.730799,3902325_id095,2024-02-24,-67.503000,-103.499490,0.303779,...,398.7545,400.987695,400.536321,-0.451374,2024,2,23,"(12.030548, -2.8300934)",12.358945,0.0
7237,WFCH20240206,2024-03-01 11:50:45,-65.76490,-60.64345,3712.493576,4902638_id004,2024-03-05,-60.798679,-64.430321,4.092535,...,425.1250,399.609350,426.936046,27.326695,2024,3,1,"(7.3894653, -3.107254)",8.016185,0.0
7238,WFCH20240303,2024-03-06 11:26:10,-55.87635,-44.48470,3717.476505,3901565_id124,2024-03-12,-45.024100,-57.620200,6.053501,...,338.7715,413.803880,339.998145,-73.805735,2024,3,6,"(-0.095825195, -2.4891968)",2.491040,0.0


In [22]:
shipDF_ice.columns

Index(['cruiseid', 'datetime', 'longitude', 'latitude', 'linear_time',
       'nearest_profid', 'prof_datetime', 'prof_lat', 'prof_lon', 'yd_sep',
       'km_sep', 'ydcos', 'ydsin', 'CT', 'SA', 'sst', 'sss', 'mld', 'adt',
       'atmos_pres_Pa', 'atmos_pres_atm', 'vapor_pres_atm', 'fco2rec',
       'pco2_atmos', 'pco2_ocean', 'delta_pco2', 'year', 'month', 'day',
       'wind_components', 'wind_speed', 'sea_ice'],
      dtype='object')

# SAVE P1-processed: ship data

p1-processed has carbon variable, mld, adt

In [23]:
shipDF_ice = pd.read_csv('../working-vars/regression/P1-processed/socatv2025_temporary_icefraction.csv')

In [26]:
use_cols = [ 'cruiseid', 'datetime',  # 'sid', 'cluster', 
    'longitude', 'latitude', 'linear_time',
    'nearest_profid', 'prof_datetime', 'prof_lat', 'prof_lon', 'yd_sep', 'km_sep', 
    'ydcos', 'ydsin',
    'CT', 'SA', 'sst', 'sss', 'mld',  
    'adt', 'wind_speed', 'sea_ice',
    'atmos_pres_Pa', 'atmos_pres_atm',  'vapor_pres_atm',
    'fco2rec', 
    'pco2_atmos', 'pco2_ocean', 'delta_pco2']

# shipDF_processed = (shipDF_ice.rename(columns={'cruiseID':'cruiseid'})
#                                 .reset_index()[use_cols].copy())

shipDF_processed = (shipDF_ice.reset_index()[use_cols].copy())

print(datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
shipDF_processed


2026-03-27 12:19:10


,cruiseid,datetime,longitude,latitude,linear_time,nearest_profid,prof_datetime,prof_lat,prof_lon,yd_sep,...,adt,wind_speed,sea_ice,atmos_pres_Pa,atmos_pres_atm,vapor_pres_atm,fco2rec,pco2_atmos,pco2_ocean,delta_pco2
0,06AQ20131221,2014-01-02 11:36:53,-24.64215,-73.72590,1.483947,7900378_id041,2014-01-09,-71.616580,-22.823985,6.720278,...,-1.2091,4.759683,0.27,99810.0,0.985048,0.005405,341.6865,385.975982,343.219907,-42.756075
1,06AQ20131221,2014-01-10 11:59:17,-36.40135,-77.10035,9.499502,7900378_id041,2014-01-09,-71.616580,-22.823985,1.295284,...,-0.9939,2.628314,0.82,99620.0,0.983173,0.005219,365.8905,385.238844,367.543164,-17.695680
2,06AQ20131221,2014-01-16 11:48:03,-38.14620,-77.89910,15.491701,7900378_id042,2014-01-19,-71.682778,-23.213771,2.700012,...,-0.9822,2.485696,0.73,99320.0,0.980212,0.005318,275.3920,383.955134,276.631562,-107.323573
3,06AQ20131221,2014-01-17 07:05:05,-38.94590,-77.60800,16.295197,7900378_id042,2014-01-19,-71.682778,-23.213771,1.896510,...,-1.0041,5.220110,0.83,99440.0,0.981396,0.005193,375.1900,384.469845,376.886195,-7.583650
4,06AQ20131221,2014-01-22 11:58:20,-28.47890,-74.55740,21.498843,7900378_id043,2014-01-29,-71.743994,-23.090656,6.695984,...,-1.2278,4.488096,0.39,101040.0,0.997187,0.005341,298.7880,390.543232,300.131879,-90.411353
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7235,WFCH20240206,2024-02-20 03:16:55,-136.38265,-73.29205,3702.136748,5906556_id041,2024-02-20,-70.554700,-135.543900,0.001024,...,-1.1179,6.558754,0.00,100310.0,0.989983,0.005426,250.8455,411.097051,251.970483,-159.126567
7236,WFCH20240206,2024-02-23 17:32:21,-99.91180,-70.46775,3705.730799,3902325_id095,2024-02-24,-67.503000,-103.499490,0.303779,...,-1.0880,12.358945,0.00,97870.0,0.965902,0.005534,398.7545,400.987695,400.536321,-0.451374
7237,WFCH20240206,2024-03-01 11:50:45,-65.76490,-60.64345,3712.493576,4902638_id004,2024-03-05,-60.798679,-64.430321,4.092535,...,-0.8436,8.016185,0.00,97710.0,0.964323,0.007141,425.1250,399.609350,426.936046,27.326695
7238,WFCH20240303,2024-03-06 11:26:10,-55.87635,-44.48470,3717.476505,3901565_id124,2024-03-12,-45.024100,-57.620200,6.053501,...,-0.0622,2.491040,0.00,102080.0,1.007451,0.016576,338.7715,413.803880,339.998145,-73.805735


In [31]:
shipDF_processed['datetime'] = pd.to_datetime(shipDF_processed['datetime'])

In [57]:
trainval = shipDF_processed[shipDF_processed.datetime.dt.year < 2024].copy()
test = shipDF_processed[shipDF_processed.datetime.dt.year == 2024].copy()

In [63]:
trainval

,cruiseid,datetime,longitude,latitude,linear_time,nearest_profid,prof_datetime,prof_lat,prof_lon,yd_sep,...,sss,mld,adt,atmos_pres_Pa,atmos_pres_atm,vapor_pres_atm,fco2rec,pco2_atmos,pco2_ocean,delta_pco2
0,06AQ20131221,2014-01-02 11:36:53,-24.64215,-73.72590,1.483947,7900378_id041,2014-01-09,-71.616580,-22.823985,6.720278,...,34.405349,11.667522,-1.2091,99810.0,0.985048,0.005405,341.6865,385.975982,343.219907,-42.756075
1,06AQ20131221,2014-01-10 11:59:17,-36.40135,-77.10035,9.499502,7900378_id041,2014-01-09,-71.616580,-22.823985,1.295284,...,34.614034,11.667522,-0.9939,99620.0,0.983173,0.005219,365.8905,385.238844,367.543164,-17.695680
2,06AQ20131221,2014-01-16 11:48:03,-38.14620,-77.89910,15.491701,7900378_id042,2014-01-19,-71.682778,-23.213771,2.700012,...,34.612038,10.548086,-0.9822,99320.0,0.980212,0.005318,275.3920,383.955134,276.631562,-107.323573
3,06AQ20131221,2014-01-17 07:05:05,-38.94590,-77.60800,16.295197,7900378_id042,2014-01-19,-71.682778,-23.213771,1.896510,...,34.752420,10.548086,-1.0041,99440.0,0.981396,0.005193,375.1900,384.469845,376.886195,-7.583650
4,06AQ20131221,2014-01-22 11:58:20,-28.47890,-74.55740,21.498843,7900378_id043,2014-01-29,-71.743994,-23.090656,6.695984,...,34.088126,10.657734,-1.2278,101040.0,0.997187,0.005341,298.7880,390.543232,300.131879,-90.411353
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6801,WFCH20231213,2023-12-15 12:29:15,-68.07260,-62.53640,3635.520312,5906560_id034,2023-12-19,-62.277300,-68.640400,4.288443,...,34.176978,50.752939,-0.8102,98320.0,0.970343,0.006176,405.2335,402.698972,407.007483,4.308512
6802,WFCH20231213,2023-12-25 13:32:16,-65.75710,-62.50970,3645.564074,5906560_id035,2023-12-30,-62.152700,-68.108100,4.586534,...,34.209054,45.531411,-0.9850,97010.0,0.957414,0.006317,404.7990,397.214424,406.563603,9.349179
6803,WFCH20231213,2023-12-26 08:47:44,-66.04485,-57.50905,3646.366481,5906560_id035,2023-12-30,-62.152700,-68.108100,3.784126,...,34.370827,45.531411,0.2052,98600.0,0.973106,0.008910,383.6670,402.619480,385.234157,-17.385323
6804,WFCH20231227,2023-12-29 17:03:41,-64.97110,-56.32710,3649.710891,6903769_id146,2024-01-04,-58.725878,-69.085745,5.560289,...,34.388894,72.831037,-0.0195,98620.0,0.973304,0.008811,386.5280,402.644772,388.110232,-14.534540


In [58]:
test

,cruiseid,datetime,longitude,latitude,linear_time,nearest_profid,prof_datetime,prof_lat,prof_lon,yd_sep,...,sss,mld,adt,atmos_pres_Pa,atmos_pres_atm,vapor_pres_atm,fco2rec,pco2_atmos,pco2_ocean,delta_pco2
6806,096U20240105,2024-01-05 12:26:19,145.71705,-44.72705,3656.518275,6903215_id255,2024-01-10,-43.790252,143.833693,5.199780,...,35.882366,27.302804,0.6885,102460.0,1.011202,0.018539,324.1865,414.543412,325.334404,-89.209007
6807,096U20240105,2024-01-06 12:06:31,142.71100,-46.66590,3657.504525,5905501_id132,2024-01-04,-45.779513,143.122653,1.525359,...,35.106165,21.520078,0.6515,101470.0,1.001431,0.013753,352.3960,412.409016,353.719551,-58.689465
6808,096U20240105,2024-01-07 11:57:49,145.40880,-49.87020,3658.498484,5906209_id143,2024-01-05,-50.733000,147.216000,1.824664,...,34.880166,92.210792,-0.0895,101160.0,0.998372,0.012037,373.2970,411.785824,374.735959,-37.049865
6809,096U20240105,2024-01-08 12:03:51,148.25500,-54.51200,3659.502674,2903453_id028,2024-01-03,-54.371100,146.883700,4.520191,...,34.258183,61.958989,-0.5517,100740.0,0.994226,0.008728,398.1170,411.410734,399.749630,-11.661104
6810,096U20240105,2024-01-09 13:07:23,149.50530,-59.49740,3660.546794,7900637_id200,2024-01-16,-59.663702,149.284939,6.884248,...,34.168063,55.597586,-1.0160,100070.0,0.987614,0.006118,360.2610,409.753503,361.840872,-47.912631
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7235,WFCH20240206,2024-02-20 03:16:55,-136.38265,-73.29205,3702.136748,5906556_id041,2024-02-20,-70.554700,-135.543900,0.001024,...,33.580067,30.405754,-1.1179,100310.0,0.989983,0.005426,250.8455,411.097051,251.970483,-159.126567
7236,WFCH20240206,2024-02-23 17:32:21,-99.91180,-70.46775,3705.730799,3902325_id095,2024-02-24,-67.503000,-103.499490,0.303779,...,33.643676,49.797021,-1.0880,97870.0,0.965902,0.005534,398.7545,400.987695,400.536321,-0.451374
7237,WFCH20240206,2024-03-01 11:50:45,-65.76490,-60.64345,3712.493576,4902638_id004,2024-03-05,-60.798679,-64.430321,4.092535,...,34.055110,42.749931,-0.8436,97710.0,0.964323,0.007141,425.1250,399.609350,426.936046,27.326695
7238,WFCH20240303,2024-03-06 11:26:10,-55.87635,-44.48470,3717.476505,3901565_id124,2024-03-12,-45.024100,-57.620200,6.053501,...,34.406736,21.728755,-0.0622,102080.0,1.007451,0.016576,338.7715,413.803880,339.998145,-73.805735


In [28]:
from datetime import datetime

In [32]:
# === Optional save: 
# newest version does preprocessing before clustering step 
# all variables at "P1" stage should have mld, adt, atmospheric pco2, deltapco2 all calculated

save = True
datetag = datetime.now().strftime('%Y%m%d') 

trainval = shipDF_processed[shipDF_processed.datetime.dt.year < 2024].copy()
test = shipDF_processed[shipDF_processed.datetime.dt.year == 2024].copy()

if save: # new version as of feb 2026 
    filepath = '../working-vars/regression/P1-processed/'

    # Split up train/test?
    filename = 'socatv2025_trainval_processed_co2_yr2014-2023_acc' + datetag + '.csv'
    trainval.to_csv(filepath + filename)

    filename = 'socatv2025_test_processed_co2_yr2024_acc' + datetag + '.csv'
    test.to_csv(filepath + filename)


    print('Saved socat training/validation data to ' + filepath + filename)

print(datetime.now().strftime('%Y-%m-%d %H:%M:%S'))


Saved socat training/validation data to ../working-vars/regression/P1-processed/socatv2025_test_processed_co2_yr2024_acc20260327.csv
2026-03-27 12:21:14
